# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset schema is specified by a Croissant JSON-LD document:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and discover records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset summary
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets with their @id and names
print("Available Record Sets:")
record_sets = metadata.record_sets
for record_set in record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# Let's list fields and columns for each record set
for record_set in record_sets:
    print(f"\nRecordSet '@id': {record_set.id}, name: {record_set.name}")
    for field in record_set.fields:
        print(f"  Field @id: {field.id}, name: {field.name}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"     Column @id: {col.id}, name: {col.name}, dataType: {col.data_type if hasattr(col, 'data_type') else 'N/A'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# Identify the main record set (manually or from the previous cell)
# For this dataset, let's pick the first record set for demonstration
if len(record_sets) == 0:
    raise Exception("No record sets found in the dataset metadata.")

main_record_set = record_sets[0]
main_record_set_id = main_record_set.id

print(f"Extracting from record set with @id: {main_record_set_id}, name: {main_record_set.name}")

# You can list all record set IDs if needed
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rsid in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rsid))
    dataframes[rsid] = df
    print(f"Loaded {df.shape[0]} records from RecordSet @id: {rsid}")

# Show available columns in the main record set
print("\nColumns in main record set DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll filter by a numeric field, normalize values, and optionally group the data. All field access is via the field's `@id` as column name.

In [ ]:
df = dataframes[main_record_set_id]

# Try to find a suitable numeric field/column by inspecting metadata
numeric_field_id = None
for field in main_record_set.fields:
    # Find a field that points to a numeric type
    if hasattr(field, 'columns'):
        for col in field.columns:
            # Use Croissant type knowledge (Float, Integer, Number, etc)
            data_type = getattr(col, 'data_type', '')
            if data_type and any(x in data_type.lower() for x in ['float', 'integer', 'number']):
                numeric_field_id = col.id
                break
    if numeric_field_id:
        break

# Fallback: Pick the first numeric-typed field
if numeric_field_id is None:
    # Try pandas type inference on first few columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if not numeric_field_id:
    raise Exception("Could not determine a numeric field for EDA.")

# Filtering, normalization, and grouping
print(f"Using numeric field '@id': {numeric_field_id}")

# Drop NA values in numeric field to allow comparison
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notna()].copy()
filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')

threshold = filtered_df[numeric_field_id].mean() if filtered_df[numeric_field_id].count() > 0 else 0
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"\nFiltered records with {numeric_field_id} > mean ({threshold:.2f}), count: {filtered_df.shape[0]}")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' (z-score normalization):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a categorical/grouping field (not numeric, not the same as the numeric field)
group_field = None
for field in main_record_set.fields:
    if hasattr(field, 'columns'):
        for col in field.columns:
            if col.id != numeric_field_id:
                # Guess at string fields
                data_type = getattr(col, 'data_type', '')
                if data_type and 'text' in data_type.lower():
                    group_field = col.id
                    break
    if group_field:
        break

# Fallback: Pick the first non-numeric pandas column
if not group_field:
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_string_dtype(df[col]):
            group_field = col
            break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Visualize a distribution and grouped means, if relevant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10,5))
    # Show mean value by group, top 20 groups sorted
    to_plot = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(data=to_plot, x=group_field, y=numeric_field_id)
    plt.xticks(rotation=60, ha='right')
    plt.title(f"Top {len(to_plot)} '{group_field}' by average {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:

- Load a complex mass spectrometry dataset described in Croissant format using `mlcroissant`
- Enumerate available record sets, fields, and columns by their `@id`
- Load records into pandas DataFrames by specifying record set `@id`
- Perform exploratory data analysis using numeric fields, applying normalization and basic grouping
- Visualize the data distributions and group means

For more detailed exploration or to work with additional record sets or metadata, revisit the 'Data Overview' section and repeat extraction steps using different `@id` values. 

This approach ensures robust, schema-aware data access and promotes reproducible workflows for complex FAIR datasets.